In [5]:
import os
import glob
import cv2
import torch
import numpy as np
from monai.networks.nets import SwinUNETR
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, Resized

In [6]:
# 1. Setup Paths
test_images_dir = "/home/jiakuny1/Projects/nnUNet_data/nnUNet_raw/Dataset101_Dental/imagesTs"
output_dir = "/home/jiakuny1/Projects/swin_predictions"
os.makedirs(output_dir, exist_ok=True)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Testing on device: {device}")

# Find all test images
test_images = sorted(glob.glob(os.path.join(test_images_dir, "*_0000.png")))
print(f"Found {len(test_images)} test images.")

Testing on device: cuda:1
Found 110 test images.


In [7]:
# 2. Load the Trained Model
model = SwinUNETR(
    in_channels=1,
    out_channels=11, 
    feature_size=24, 
    spatial_dims=2
).to(device)

# Load the Epoch 100 weights
weights_path = "/home/jiakuny1/Projects/swin_unetr_epoch_100.pth"
model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
model.eval() # Lock the weights for testing!

SwinUNETR(
  (swinViT): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(1, 24, kernel_size=(2, 2), stride=(2, 2))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers1): ModuleList(
      (0): BasicLayer(
        (blocks): ModuleList(
          (0-1): 2 x SwinTransformerBlock(
            (norm1): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=24, out_features=72, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=24, out_features=24, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path): Identity()
            (norm2): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
            (mlp): MLPBlock(
              (linear1): Linear(in_features=24, out_features=96, bias=True)
              (linear2): Linear(in_features=96, ou

In [8]:
# 3. Define the exact same transform used for training (but no labels this time)
test_transform = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityd(keys=["image"]),
    Resized(keys=["image"], spatial_size=(512, 512), mode="bilinear"),
])

print("Running inference...")

with torch.no_grad(): # Disable gradient calculation to save memory
    for img_path in test_images:
        filename = os.path.basename(img_path).replace("_0000.png", ".png")
        
        # We need the original image shape so we can resize the prediction back to normal
        original_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        original_h, original_w = original_img.shape
        
        # Prepare the input dictionary for MONAI transforms
        data = {"image": img_path}
        transformed = test_transform(data)
        
        # Add a batch dimension [1, C, H, W] and send to GPU
        input_tensor = transformed["image"].unsqueeze(0).to(device)
        
        # Generate prediction
        output = model(input_tensor)
        
        # Get the highest probability class for each pixel (argmax)
        prediction = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)
        
        # Resize the 512x512 prediction back to the original X-Ray dimensions
        # CRITICAL: We MUST use INTER_NEAREST so it doesn't blend class numbers (e.g. 1 and 3 blending into a fake class 2)
        prediction_resized = cv2.resize(prediction, (original_w, original_h), interpolation=cv2.INTER_NEAREST)
        
        # Save the mask
        save_path = os.path.join(output_dir, filename)
        cv2.imwrite(save_path, prediction_resized)
        
        print(f"Saved prediction for {filename}")

print("\nAll testing complete! Masks saved to:", output_dir)

Running inference...
Saved prediction for IO000001_(1013).png
Saved prediction for IO000001_(109).png
Saved prediction for IO000001_(112).png
Saved prediction for IO000001_(114).png
Saved prediction for IO000001_(116).png
Saved prediction for IO000001_(118).png
Saved prediction for IO000001_(12).png
Saved prediction for IO000001_(120).png
Saved prediction for IO000001_(122).png
Saved prediction for IO000001_(125).png
Saved prediction for IO000001_(128).png
Saved prediction for IO000001_(130).png
Saved prediction for IO000001_(14).png
Saved prediction for IO000001_(150).png
Saved prediction for IO000001_(170).png
Saved prediction for IO000001_(175).png
Saved prediction for IO000001_(176).png
Saved prediction for IO000001_(185).png
Saved prediction for IO000001_(186).png
Saved prediction for IO000001_(189).png
Saved prediction for IO000001_(19).png
Saved prediction for IO000001_(197).png
Saved prediction for IO000001_(20).png
Saved prediction for IO000001_(208).png
Saved prediction for I